# XP Exercises: LLM Summarization Evaluation
Hands-on tutorial for evaluating LLMs on summarization tasks using accuracy and ROUGE metrics, model comparison, and custom functions.

In [ ]:
# Part I. Setup
%pip install rouge_score==0.1.2
%pip install evaluate
%pip install -U accelerate --quiet
%pip install datasets
%pip install nltk
import nltk
nltk.download('punkt')

In [ ]:
# Part II. Dataset Loading and Exploration
import pandas as pd
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
train_sample = train_df.sample(n=100, random_state=42).reset_index(drop=True)
test_sample = test_df.sample(n=50, random_state=42).reset_index(drop=True)
print('First train example:')
print('Article:', train_sample.loc[0, 'prompt_text'])
print('Reference summary:', train_sample.loc[0, 'prompt_title'])
print('Train sample DataFrame:')
print(train_sample.head())
print('Test sample DataFrame:')
print(test_sample.head())

In [ ]:
# Part III. Summarization with T5
from transformers import T5ForConditionalGeneration, AutoTokenizer
import torch, gc
def batch_generator(data, batch_size):
    for i in range(0, len(data), batch_size):
        yield data[i:i+batch_size]
def summarize_with_t5(texts, model_name='t5-small', batch_size=8):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)
    summaries = []
    for batch in batch_generator(texts, batch_size):
        inputs = tokenizer(['summarize: ' + t for t in batch], return_tensors='pt', padding=True, truncation=True).to(device)
        outputs = model.generate(**inputs, max_new_tokens=32)
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        summaries.extend(decoded)
        torch.cuda.empty_cache(); gc.collect()
    torch.cuda.empty_cache(); gc.collect()
    return summaries
t5_summaries = summarize_with_t5(train_sample['prompt_text'].tolist())
pd.DataFrame({'Reference': train_sample['prompt_title'], 'T5 Summary': t5_summaries}).head()

In [ ]:
# Part IV. Accuracy Evaluation
def compute_accuracy(preds, refs):
    return sum([p.strip() == r.strip() for p, r in zip(preds, refs)]) / len(refs)
acc = compute_accuracy(t5_summaries, train_sample['prompt_title'].tolist())
print(f'Accuracy: {acc:.4f}')
# Accuracy is expected to be very low for summarization tasks.

In [ ]:
# Part V. ROUGE Metric Implementation
import evaluate
rouge = evaluate.load('rouge')
def preprocess_for_rouge(text):
    return '
'.join(nltk.sent_tokenize(text))
def compute_rouge_score(preds, refs):
    preds = [preprocess_for_rouge(p) for p in preds]
    refs = [preprocess_for_rouge(r) for r in refs]
    return rouge.compute(predictions=preds, references=refs)

In [ ]:
# Part VI. Understanding ROUGE Scores
identical_preds = train_sample['prompt_title'].tolist()
empty_preds = [''] * len(identical_preds)
print('ROUGE (identical):', compute_rouge_score(identical_preds, identical_preds))
print('ROUGE (empty):', compute_rouge_score(empty_preds, identical_preds))
# Stemming and n-gram analysis
print('ROUGE-1:', rouge.compute(predictions=['cats are running'], references=['cat runs']))
print('ROUGE-2:', rouge.compute(predictions=['cats are running fast'], references=['cat runs quickly']))

In [ ]:
# Part VII. Comparing Small and Large Models
def summarize_with_gpt2(texts, model_name='gpt2', batch_size=8):
    from transformers import GPT2LMHeadModel, AutoTokenizer
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = GPT2LMHeadModel.from_pretrained(model_name).to(device)
    summaries = []
    for batch in batch_generator(texts, batch_size):
        inputs = tokenizer(['TL;DR: ' + t for t in batch], return_tensors='pt', padding=True, truncation=True).to(device)
        outputs = model.generate(**inputs, max_new_tokens=32)
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        summaries.extend([d.split('TL;DR: ')[-1] for d in decoded])
        torch.cuda.empty_cache(); gc.collect()
    torch.cuda.empty_cache(); gc.collect()
    return summaries
t5_base_summaries = summarize_with_t5(train_sample['prompt_text'].tolist(), model_name='t5-base')
gpt2_summaries = summarize_with_gpt2(train_sample['prompt_text'].tolist())

In [ ]:
def compute_rouge_per_row(preds, refs):
    scores = [rouge.compute(predictions=[preprocess_for_rouge(p)], references=[preprocess_for_rouge(r)]) for p, r in zip(preds, refs)]
    return pd.DataFrame(scores)
print('T5-small per-row ROUGE:')
print(compute_rouge_per_row(t5_summaries, train_sample['prompt_title'].tolist()).head())
print('T5-base per-row ROUGE:')
print(compute_rouge_per_row(t5_base_summaries, train_sample['prompt_title'].tolist()).head())
print('GPT2 per-row ROUGE:')
print(compute_rouge_per_row(gpt2_summaries, train_sample['prompt_title'].tolist()).head())

In [ ]:
# Part VIII. Comparing All Models
def compare_models(*model_summaries, refs):
    results = {}
    for i, summaries in enumerate(model_summaries):
        scores = compute_rouge_score(summaries, refs)
        results[f'Model_{i+1}'] = scores
    return pd.DataFrame(results)

In [ ]:
def compare_models_summaries(refs, *model_summaries):
    df = pd.DataFrame({'Reference': refs})
    for i, summaries in enumerate(model_summaries):
        df[f'Model_{i+1}_Summary'] = summaries
    return df.head()
agg_scores = compare_models(t5_summaries, t5_base_summaries, gpt2_summaries, refs=train_sample['prompt_title'].tolist())
print('Aggregated ROUGE scores:')
print(agg_scores)
print('Side-by-side summary comparison:')
print(compare_models_summaries(train_sample['prompt_title'].tolist(), t5_summaries, t5_base_summaries, gpt2_summaries))